In [3]:
# --- MASTER INTERACTIVE GUI (WITH DUAL SLIDER & TEXT BOX INPUTS) ---
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, clear_output

%matplotlib inline

# Global Database Configurations
DB_NAME = "payroll_ledger.db"
CSV_BACKUP_NAME = "payroll_history_backup.csv"
EMPLOYEE_ID = "EE-SCORP-01"

# 2026 Regulatory Cap Metrics
SS_WAGE_BASE = 184500.00      
FUTA_WAGE_BASE = 7000.00      
NM_SUI_WAGE_BASE = 34800.00   
SS_RATE, MEDICARE_RATE, FUTA_NET_RATE, NM_SUI_RATE = 0.062, 0.0145, 0.006, 0.034

# 2026 Progressive Income Tax Tables 
TAX_TABLES_2026 = {
    'Single': {
        'FED': [(12400, 0.10), (50400, 0.12), (105700, 0.22), (201775, 0.24), (256225, 0.32), (640600, 0.35), (float('inf'), 0.37)],
        'NM':  [(8050, 0.00), (13550, 0.015), (20550, 0.032), (24550, 0.032), (33000, 0.043), (41000, 0.043), (58000, 0.047), (74000, 0.047), (102100, 0.047), (116100, 0.049), (217500, 0.049), (331100, 0.049), (float('inf'), 0.059)]
    },
    'Married Filing Jointly': {
        'FED': [(24800, 0.10), (100800, 0.12), (211400, 0.22), (403550, 0.24), (512450, 0.32), (768700, 0.35), (float('inf'), 0.37)],
        'NM':  [(16100, 0.00), (27100, 0.015), (41100, 0.032), (49100, 0.032), (66000, 0.043), (82000, 0.043), (116000, 0.047), (148000, 0.047), (204200, 0.047), (232200, 0.049), (435000, 0.049), (662200, 0.049), (float('inf'), 0.059)]
    }
}

# --- 1. SETUP DUAL-INPUT FORM CONTROLS ---
# Setup consistent text spacing widths
lbl_style = widgets.Layout(width='110px')

status_dropdown = widgets.Dropdown(options=['Single', 'Married Filing Jointly'], value='Single', description='Filing Status:', style={'description_width': 'initial'})
date_picker = widgets.DatePicker(value=pd.to_datetime('2026-06-15').date(), description='Pay Date:', style={'description_width': 'initial'})

# Pair A: Annual Base Salary
salary_lbl = widgets.Label('Annual Base:', layout=lbl_style)
salary_slide = widgets.IntSlider(value=60000, min=30000, max=250000, step=1000, continuous_update=False)
salary_box = widgets.IntText(value=60000, layout=widgets.Layout(width='90px'))
widgets.jslink((salary_slide, 'value'), (salary_box, 'value')) # 🔗 Link slider & box
salary_ui = widgets.HBox([salary_lbl, salary_slide, salary_box])

# Pair B: Pay Periods
periods_lbl = widgets.Label('Pay Periods:', layout=lbl_style)
periods_slide = widgets.IntSlider(value=12, min=12, max=52, step=1, continuous_update=False)
periods_box = widgets.IntText(value=12, layout=widgets.Layout(width='90px'))
widgets.jslink((periods_slide, 'value'), (periods_box, 'value')) # 🔗 Link slider & box
periods_ui = widgets.HBox([periods_lbl, periods_slide, periods_box])

# Pair C: S-Corp Health Insurance Premium (HIP)
hip_lbl = widgets.Label('S-Corp HIP:', layout=lbl_style)
hip_slide = widgets.FloatSlider(value=1310.0, min=0.0, max=2000.0, step=10.0, continuous_update=False)
hip_box = widgets.FloatText(value=1310.0, layout=widgets.Layout(width='90px'))
widgets.jslink((hip_slide, 'value'), (hip_box, 'value')) # 🔗 Link slider & box
hip_ui = widgets.HBox([hip_lbl, hip_slide, hip_box])

# Pair D: 401k Deferral Percentage
retire_lbl = widgets.Label('401k Def %:', layout=lbl_style)
retire_slide = widgets.FloatSlider(value=0.0, min=0.0, max=25.0, step=0.5, continuous_update=False)
retire_box = widgets.FloatText(value=0.0, layout=widgets.Layout(width='90px'))
widgets.jslink((retire_slide, 'value'), (retire_box, 'value')) # 🔗 Link slider & box
retire_ui = widgets.HBox([retire_lbl, retire_slide, retire_box])

save_button = widgets.Button(description="Commit Pay Run", button_style='success', icon='check')
output_panel = widgets.Output()

# --- 2. COMPUTATIONAL MATH FUNCTIONS ---
def calc_capped(gross, ytd, cap, rate):
    if ytd >= cap: return 0.0
    return min(gross, cap - ytd) * rate

def calc_progressive(gross, periods, brackets):
    annual = gross * periods
    tax = 0.0
    prev = 0.0
    for limit, rate in brackets:
        if annual > limit:
            tax += (limit - prev) * rate
            prev = limit
        else:
            tax += (annual - prev) * rate
            break
    return tax / periods

# --- 3. DASHBOARD REFRESH LOOP ---
def update_dashboard(*args):
    with output_panel:
        clear_output(wait=True)
        
        # Calculate dynamic YTD metrics from live DB
        conn = sqlite3.connect(DB_NAME)
        cursor = conn.cursor()
        cursor.execute("SELECT SUM(gross_wages) FROM payroll_ledger WHERE employee_id = ?", (EMPLOYEE_ID,))
        res = cursor.fetchone()
        ytd_prior = float(res[0]) if res and res[0] is not None else 0.00
        conn.close()
        
        # Pull parameters safely from the shared interactive sync values
        status = status_dropdown.value
        periods = periods_slide.value
        base_gross = salary_slide.value / periods
        hip_reimbursement = hip_slide.value
        
        total_period_payout = base_gross + hip_reimbursement
        deduction_401k = base_gross * (retire_slide.value / 100.0)
        
        # S-Corp tax optimization rule separation
        fica_subject_gross = base_gross 
        income_tax_subject_gross = base_gross + hip_reimbursement - deduction_401k
        
        # Run tax calculations
        ee_ss = calc_capped(fica_subject_gross, ytd_prior, SS_WAGE_BASE, SS_RATE)
        ee_med = fica_subject_gross * MEDICARE_RATE
        ee_fed = calc_progressive(income_tax_subject_gross, periods, TAX_TABLES_2026[status]['FED'])
        ee_nm = calc_progressive(income_tax_subject_gross, periods, TAX_TABLES_2026[status]['NM'])
        
        nm_wc_fee_per_pay = (2.55 * 4) / periods
        total_ee_taxes = ee_ss + ee_med + ee_fed + ee_nm + nm_wc_fee_per_pay
        net_pay = total_period_payout - deduction_401k - total_ee_taxes
        
        # Render Pie Chart Visualization
        categories = ['Net Take-Home', 'Pre-Tax 401k', 'Fed Income Tax', 'NM State Tax', 'FICA taxes']
        sizes = [net_pay, deduction_401k, ee_fed, ee_nm, (ee_ss + ee_med)]
        colors = ['#2ecc71', '#3498db', '#e74c3c', '#f1c40f', '#95a5a6']
        
        combined_labels = [f"{cat}\n(${val:,.2f})" for cat, val in zip(categories, sizes)]
        labels = [l for i, l in enumerate(combined_labels) if sizes[i] > 0]
        colors = [c for i, c in enumerate(colors) if sizes[i] > 0]
        sizes = [s for s in sizes if s > 0]
        
        fig, ax = plt.subplots(figsize=(8.5, 4.5))
        ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=140, pctdistance=0.65, labeldistance=1.15, textprops={'fontsize': 9, 'weight': 'bold'})
        ax.set_title(f"2026 S-Corp Payout Breakdown [{status.upper()}] (Gross Total: ${total_period_payout:,.2f})", fontsize=11, weight='bold', pad=20)
        plt.tight_layout()
        plt.show()
        
        print(f"💰 Owner Net Cash: ${net_pay:,.2f} | 🩺 Taxable S-Corp HIP Amount: ${hip_reimbursement:,.2f}")
        print(f"ℹ️ Base Wages (FICA Subject): ${base_gross:,.2f} | Combined Tax Subject Base: ${income_tax_subject_gross:,.2f}")

# --- 4. ACTION HANDLE: COMMIT TO DATABASE ---
def on_save_clicked(b):
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute("SELECT SUM(gross_wages) FROM payroll_ledger WHERE employee_id = ?", (EMPLOYEE_ID,))
    res = cursor.fetchone()
    ytd_prior = float(res) if res and res is not None else 0.00
    
    status = status_dropdown.value
    periods = periods_slide.value
    base_gross = salary_slide.value / periods
    hip_reimbursement = hip_slide.value
    current_gross = base_gross + hip_reimbursement
    deduction_401k = base_gross * (retire_slide.value / 100.0)
    
    ee_ss = calc_capped(base_gross, ytd_prior, SS_WAGE_BASE, SS_RATE)
    ee_med = base_gross * MEDICARE_RATE
    ee_fed = calc_progressive(base_gross + hip_reimbursement - deduction_401k, periods, TAX_TABLES_2026[status]['FED'])
    ee_nm = calc_progressive(base_gross + hip_reimbursement - deduction_401k, periods, TAX_TABLES_2026[status]['NM'])
    nm_wc_fee = (2.55 * 4) / periods
    net_pay = current_gross - deduction_401k - (ee_ss + ee_med + ee_fed + ee_nm + nm_wc_fee)
    
    cursor.execute("""
    INSERT INTO payroll_ledger (
        employee_id, pay_period_ending, gross_wages, pre_tax_401k, ee_ss, ee_medicare, ee_fed_tax, ee_nm_tax, ee_wc_fee, net_pay,
        er_ss, er_medicare, er_f展a, er_nm_sui, er_wc_fee, total_company_cost
    ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    """, (
        EMPLOYEE_ID, str(date_picker.value), current_gross, deduction_401k, ee_ss, ee_med, ee_fed, ee_nm, nm_wc_fee, net_pay,
        ee_ss, ee_med, calc_capped(base_gross, ytd_prior, FUTA_WAGE_BASE, FUTA_NET_RATE), calc_capped(base_gross, ytd_prior, NM_SUI_WAGE_BASE, NM_SUI_RATE), nm_wc_fee, (current_gross + ee_ss + ee_med + nm_wc_fee)
    ))
    conn.commit()
    df = pd.read_sql_query("SELECT * FROM payroll_ledger", conn)
    df.to_csv(CSV_BACKUP_NAME, index=False)
    conn.close()
    
    update_dashboard()
    with output_panel:
        print(f"\n✅ SUCCESS: Payroll committed safely.")

# Bind standard value observers to trigger updates instantly
status_dropdown.observe(update_dashboard, 'value')
salary_slide.observe(update_dashboard, 'value')
periods_slide.observe(update_dashboard, 'value')
hip_slide.observe(update_dashboard, 'value')
retire_slide.observe(update_dashboard, 'value')
date_picker.observe(update_dashboard, 'value')
save_button.on_click(on_save_clicked)

# Render Combined Form Controls Group Layout Layout
input_form = widgets.VBox([status_dropdown, salary_ui, periods_ui, hip_ui, retire_ui, date_picker, save_button])
dashboard_layout = widgets.HBox([input_form, output_panel])
display(dashboard_layout)

update_dashboard()
